In [1]:
import pandas as pd
import numpy as np
import re

from extraction.texts import model_loader, text_encoding

In [2]:
df = pd.read_csv('data/texts/suicide_harm/suicide.csv')
df = df[["text", 'class']]

df['class'] = df['class'].replace({
                            "suicide":1,
                            "non-suicide":0
                        })

df = pd.concat([
    df[df["class"] == 0].head(10000),
    df[df["class"] == 1].head(40000)
]).reset_index(drop=True)

df["text"] = (
    df["text"].apply(lambda x: re.sub(r"[^A-Za-z0-9]", " ", 
                                             x, count=0, flags=0))
            .apply(lambda x: re.findall(r"[A-Za-z0-9]+", x))
            .apply(lambda x: " ".join(x))
)

/var/folders/tg/66tmmx452mg3qln33ts5xqh80000gn/T/ipykernel_9909/4066350544.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['class'] = df['class'].replace({


In [3]:
texts = df["text"].to_list()

In [4]:
model, model_code = model_loader('mps', model_code='e5')
text_vectors = text_encoding(texts, model, model_code)

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

In [5]:
from sklearn.model_selection import train_test_split
from training.kfold_train import stratified_kfold_train_val

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.metrics import confusion_matrix, classification_report

In [6]:
x, y = text_vectors, df["class"].to_numpy()
x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    stratify=y,
                                                    test_size=0.2)

In [7]:
log_reg = LogisticRegression(
    penalty='l1',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='saga',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)

stratified_kfold_train_val(5,
                           0.35,
                           log_reg,
                           x_train,
                           y_train)

Starting 5-Fold Stratified Cross-Validation...

sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9782 (When flagged positive, accuracy is 97.82%)
Custom Recall Score:    0.9598 (Captured 95.98% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9730 (When flagged positive, accuracy is 97.30%)
Custom Recall Score:    0.9575 (Captured 95.75% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9745 (When flagged positive, accuracy is 97.45%)
Custom Recall Score:    0.9684 (Captured 96.84% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9760 (When flagged positive, accuracy is 97.60%)
Custom Recall Score:    0.9613 (Captured 96.12% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9743 (When flagged positive, accuracy is 97.43%)
Custom R

In [8]:
log_reg = LogisticRegression(
    penalty='l1',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='saga',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)
threshold = 0.35

log_reg.fit(x_train, y_train)
test_probabilities = log_reg.predict_proba(x_test)[:, 1]
predictions = (test_probabilities >= threshold).astype(int)

print(classification_report(y_test, predictions))
print(confusion_matrix(y_test, predictions))

              precision    recall  f1-score   support

           0       0.86      0.90      0.88      2000
           1       0.98      0.96      0.97      8000

    accuracy                           0.95     10000
   macro avg       0.92      0.93      0.92     10000
weighted avg       0.95      0.95      0.95     10000

[[1807  193]
 [ 297 7703]]


In [9]:
import joblib

In [10]:
joblib.dump(log_reg, open("model/suicide.jobllib", 'wb'))